<a href="https://colab.research.google.com/github/engmodu/AIFEL_quest_eng/blob/main/LLM_Application/LLM01/Day1_RAG_Code_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [42]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-5",
    temperature=0.0,
)
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/bible_pdf/1-20잠언.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/bible_pdf/1-20잠언.pdf")
pages = loader.load_and_split()

print("현재 작업 경로 :", os.getcwd())
print("파일 존재 여부 :", os.path.exists(file_path))
print("파일 여부 :", os.path.isfile(file_path))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 작업 경로 : /content
파일 존재 여부 : True
파일 여부 : True


In [44]:
print(pages[10].page_content)

잠14:26 여호와를 경외하는 자에게는 견고한 의뢰가 있나니 그 자녀들에게 피난처가 있으리라잠14:27 여호와를 경외하는 것은 생명의 샘이니 사망의 그물에서 벗어나게 하느니라잠14:28 백성이 많은 것은 왕의 영광이요 백성이 적은 것은 주권자의 패망이니라잠14:29 노하기를 더디 하는 자는 크게 명철하여도 마음이 조급한 자는 어리석음을 나타내느니라잠14:30 평온한 마음은 육신의 생명이나 시기는 뼈를 썩게 하느니라잠14:31 가난한 사람을 학대하는 자는 그를 지으신 이를 멸시하는 자요 궁핍한 사람을 불쌍히 여기는 자는 주를 공경하는 자니라잠14:32 악인은 그의 환난에 엎드러져도 의인은 그의 죽음에도 소망이 있느니라잠14:33 지혜는 명철한 자의 마음에 머물거니와 미련한 자의 속에 있는 것은 나타나느니라잠14:34 공의는 나라를 영화롭게 하고 죄는 백성을 욕되게 하느니라잠14:35 슬기롭게 행하는 신하는 왕에게 은총을 입고 욕을 끼치는 신하는 그의 진노를 당하느니라잠15:1 유순한 대답은 분노를 쉬게 하여도 과격한 말은 노를 격동하느니라잠15:2 지혜 있는 자의 혀는 지식을 선히 베풀고 미련한 자의 입은 미련한 것을 쏟느니라잠15:3 여호와의 눈은 어디서든지 악인과 선인을 감찰하시느니라잠15:4 온순한 혀는 곧 생명 나무이지만 패역한 혀는 마음을 상하게 하느니라잠15:5 아비의 훈계를 업신여기는 자는 미련한 자요 경계를 받는 자는 슬기를 얻을 자니라잠15:6 의인의 집에는 많은 보물이 있어도 악인의 소득은 고통이 되느니라잠15:7 지혜로운 자의 입술은 지식을 전파하여도 미련한 자의 마음은 정함이 없느니라잠15:8 악인의 제사는 여호와께서 미워하셔도 정직한 자의 기도는 그가 기뻐하시느니라잠15:9 악인의 길은 여호와께서 미워하셔도 공의를 따라가는 자는 그가 사랑하시느니라잠15:10 도를 배반하는 자는 엄한 징계를 받을 것이요 견책을 싫어하는 자는 죽을 것이니라잠15:11 스올과 아바돈도 여호와의 앞에 드러나거든 하물며 사람의 마음이리요잠15:12 거만한 자는 견

Text splitter 사용을 위한 준비입니다

### Step 1 Document loader

In [46]:
file_path = home_path + "/data/bible_txt/2-03누가복음.txt"

encodings = ["utf-8", "cp949", "euc-kr"]

for enc in encodings:
    try:
        with open(file_path, encoding=enc) as f:
            text = f.read()

        print(f"성공 인코딩 : {enc}")
        print(text[:300])
        break

    except Exception as e:
        print(f"{enc} 실패 :", e)

utf-8 실패 : 'utf-8' codec can't decode byte 0xb4 in position 0: invalid start byte
성공 인코딩 : cp949
눅1:1 <데오빌로 각하에게> 우리 중에 이루어진 사실에 대하여
눅1:2 처음부터 목격자와 말씀의 일꾼 된 자들이 전하여 준 그대로 내력을 저술하려고 붓을 든 사람이 많은지라
눅1:3 그 모든 일을 근원부터 자세히 미루어 살핀 나도 데오빌로 각하에게 차례대로 써 보내는 것이 좋은 줄 알았노니
눅1:4 이는 각하가 알고 있는 바를 더 확실하게 하려 함이로라
눅1:5 <세례 요한의 출생을 예고하다> 유대 왕 헤롯 때에 아비야 반열에 제사장 한 사람이 있었으니 이름은 사가랴요 그의 아내는 아론의 자손이니 이름은 엘리사벳이라
눅1:6 이 


In [59]:
from langchain_text_splitters import CharacterTextSplitter
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=1000, length_function = len,)
chunks = text_splitter.split_text(text)

In [60]:
print(chunks[0])

눅1:1 <데오빌로 각하에게> 우리 중에 이루어진 사실에 대하여
눅1:2 처음부터 목격자와 말씀의 일꾼 된 자들이 전하여 준 그대로 내력을 저술하려고 붓을 든 사람이 많은지라
눅1:3 그 모든 일을 근원부터 자세히 미루어 살핀 나도 데오빌로 각하에게 차례대로 써 보내는 것이 좋은 줄 알았노니
눅1:4 이는 각하가 알고 있는 바를 더 확실하게 하려 함이로라
눅1:5 <세례 요한의 출생을 예고하다> 유대 왕 헤롯 때에 아비야 반열에 제사장 한 사람이 있었으니 이름은 사가랴요 그의 아내는 아론의 자손이니 이름은 엘리사벳이라
눅1:6 이 두 사람이 하나님 앞에 의인이니 주의 모든 계명과 규례대로 흠이 없이 행하더라
눅1:7 엘리사벳이 잉태를 못하므로 그들에게 자식이 없고 두 사람의 나이가 많더라
눅1:8 마침 사가랴가 그 반열의 차례대로 하나님 앞에서 제사장의 직무를 행할새
눅1:9 제사장의 전례를 따라 제비를 뽑아 주의 성전에 들어가 분향하고
눅1:10 모든 백성은 그 분향하는 시간에 밖에서 기도하더니
눅1:11 주의 사자가 그에게 나타나 향단 우편에 선지라
눅1:12 사가랴가 보고 놀라며 무서워하니
눅1:13 천사가 그에게 이르되 사가랴여 무서워하지 말라 너의 간구함이 들린지라 네 아내 엘리사벳이 네게 아들을 낳아 주리니 그 이름을 요한이라 하라
눅1:14 너도 기뻐하고 즐거워할 것이요 많은 사람도 그의 태어남을 기뻐하리니
눅1:15 이는 그가 주 앞에 큰 자가 되며 포도주나 독한 술을 마시지 아니하며 모태로부터 성령의 충만함을 받아
눅1:16 이스라엘 자손을 주 곧 그들의 하나님께로 많이 돌아오게 하겠음이라
눅1:17 그가 또 엘리야의 심령과 능력으로 주 앞에 먼저 와서 아버지의 마음을 자식에게, 거스르는 자를 의인의 슬기에 돌아오게 하고 주를 위하여 세운 백성을 준비하리라
눅1:18 사가랴가 천사에게 이르되 내가 이것을 어떻게 알리요 내가 늙고 아내도 나이가 많으니이다


In [61]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[948, 984, 993, 979, 974, 987, 969, 988, 985, 986, 981, 990, 989, 942, 976, 972, 967, 983, 987, 998, 992, 971, 986, 988, 988, 974, 975, 986, 979, 983, 970, 948, 966, 996, 975, 989, 973, 971, 1000, 997, 997, 956, 962, 997, 983, 989, 971, 958, 975, 986, 990, 999, 999, 987, 963, 990, 988, 974, 974, 993, 1000, 969, 972, 977, 985, 985, 974, 953, 972, 963, 982, 957, 971, 973, 967, 986, 994, 996, 979, 969, 989, 990, 982, 976, 973, 962, 993, 993, 996, 969, 980, 998, 989, 965, 988, 985, 971, 995, 994, 957, 974, 973, 974, 964, 978, 970, 975, 940, 989, 970, 989, 967, 958, 956, 978, 964, 973, 997, 967, 974, 963, 951, 967, 951, 972, 999, 982, 979, 969, 952, 993, 996, 999, 995, 985, 963, 993, 982, 969, 968, 993, 967, 969, 973, 936, 948, 940, 973, 986, 988, 996, 982, 988, 981, 946, 995, 988, 989, 995, 945, 958, 998, 963, 997, 984, 952, 985, 986, 950, 981, 961, 992, 960, 986, 969, 969, 967, 966, 979, 984, 970, 973, 985, 940, 999, 957, 964, 992, 994, 988, 970, 979, 931, 1000, 999, 976, 980, 950, 946, 9

### Step 2 Text splitters

In [62]:
!pip install tiktoken

In [63]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[948, 984, 993, 979, 974, 987, 969, 988, 985, 986, 981, 990, 989, 942, 976, 972, 967, 983, 987, 998, 992, 971, 986, 988, 988, 974, 975, 986, 979, 983, 970, 948, 966, 996, 975, 989, 973, 971, 1000, 997, 997, 956, 962, 997, 983, 989, 971, 958, 975, 986, 990, 999, 999, 987, 963, 990, 988, 974, 974, 993, 1000, 969, 972, 977, 985, 985, 974, 953, 972, 963, 982, 957, 971, 973, 967, 986, 994, 996, 979, 969, 989, 990, 982, 976, 973, 962, 993, 993, 996, 969, 980, 998, 989, 965, 988, 985, 971, 995, 994, 957, 974, 973, 974, 964, 978, 970, 975, 940, 989, 970, 989, 967, 958, 956, 978, 964, 973, 997, 967, 974, 963, 951, 967, 951, 972, 999, 982, 979, 969, 952, 993, 996, 999, 995, 985, 963, 993, 982, 969, 968, 993, 967, 969, 973, 936, 948, 940, 973, 986, 988, 996, 982, 988, 981, 946, 995, 988, 989, 995, 945, 958, 998, 963, 997, 984, 952, 985, 986, 950, 981, 961, 992, 960, 986, 969, 969, 967, 966, 979, 984, 970, 973, 985, 940, 999, 957, 964, 992, 994, 988, 970, 979, 931, 1000, 999, 976, 980, 950, 946, 9

### Step 3 Vector Empeddings

In [64]:
import openai
client = openai.OpenAI()
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
!curl ipinfo.io

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large
{
  "ip": "35.186.157.41",
  "hostname": "41.157.186.35.bc.googleusercontent.com",
  "city": "Singapore",
  "region": "Singapore",
  "country": "SG",
  "loc": "1.2897,103.8501",
  "org": "AS396982 Google LLC",
  "postal": "018989",
  "timezone": "Asia/Singapore",
  "readme": "https://ipinfo.io/missingauth"
}

In [73]:
embeddings = embedding_model.embed_documents(
    [
        "예수님",
        "마리아",
        "성령",
    ]
)
print(embeddings[1])
len(embeddings[1])

[-0.0180206298828125, 0.004985809326171875, -0.01181793212890625, -0.03887939453125, -0.06890869140625, 0.0341796875, 0.01873779296875, 0.042449951171875, -0.031646728515625, 0.00222015380859375, -0.030059814453125, 0.0294189453125, 0.005558013916015625, -0.0082855224609375, 0.004486083984375, 0.01374053955078125, -0.006565093994140625, -0.042388916015625, 0.05230712890625, -0.0149688720703125, 0.0290985107421875, 0.04473876953125, -0.0096282958984375, 0.004634857177734375, 0.00699615478515625, 0.067138671875, -0.0231781005859375, -0.0116424560546875, 0.036865234375, 0.04022216796875, 0.035064697265625, -0.03375244140625, 0.049346923828125, 0.0095977783203125, -0.00041031837463378906, 0.0017404556274414062, 0.016876220703125, -0.031280517578125, 0.01042938232421875, 0.0246429443359375, -0.04150390625, -0.005725860595703125, 0.011444091796875, 0.0027980804443359375, 0.019012451171875, 0.03570556640625, -0.031585693359375, 0.005931854248046875, 0.0231475830078125, 0.006542205810546875, -

1536

In [75]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [83]:
query = ["백성"]
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.2347460615161856
0.1543533999124861
0.35105541123620204


In [1]:
#!pip install chromadb
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -U opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -U langchain-chroma chromadb
!pip install langchain-chroma
from langchain_chroma import Chroma
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk
!pip show chromadb
# 위에서 사용했던 코드입니다
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from google.colab import userdata
from langchain_openai import ChatOpenAI
import os

!pip install -U langchain-openai openai
!pip install -U sentence-transformers

def tiktoken_len(text):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    return len(tokens)

from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/bible_pdf/1-20잠언.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/bible_pdf/1-20잠언.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

from langchain_community.embeddings import HuggingFaceEmbeddings

Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: opentelemetry-semantic-conventions 0.63b1
Uninstalling opentelemetry-semantic-conventions-0.63b1:
  Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl (203 kB)
ERROR: pip's dependency resolver does not currently take into ac

/tmp/ipykernel_33038/3186609261.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
# import
import os

from google.colab import userdata

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

# OpenAI Key
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

# PDF 로드
from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/bible_pdf/1-20잠언.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/bible_pdf/1-20잠언.pdf")
pages = loader.load_and_split()

print("페이지 수 :", len(pages))

# Chunk 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = text_splitter.split_documents(pages)

print("Chunk 수 :", len(docs))
print(docs[0].page_content[:300])

# Embedding 모델
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Chroma Vector DB 생성
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

print("Chroma DB 생성 완료")

페이지 수 : 24
Chunk 수 : 51
잠1:1 <솔로몬의 잠언> 다윗의 아들 이스라엘 왕 솔로몬의 잠언이라잠1:2 이는 지혜와 훈계를 알게 하며 명철의 말씀을 깨닫게 하며잠1:3 지혜롭게, 공의롭게, 정의롭게, 정직하게 행할 일에 대하여 훈계를 받게 하며잠1:4 어리석은 자를 슬기롭게 하며 젊은 자에게 지식과 근신함을 주기 위한 것이니잠1:5 지혜 있는 자는 듣고 학식이 더할 것이요 명철한 자는 지략을 얻을 것이라잠1:6 잠언과 비유와 지혜 있는 자의 말과 그 오묘한 말을 깨달으리라잠1:7 <젊은이에게 주는 교훈> 여호와를 경외하는 것이 지식의 근본이거늘 미련한 자는 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chroma DB 생성 완료


In [21]:
query = "밥먹자"
docs = db.similarity_search(query)

In [22]:
print(docs[0].page_content)

악인을 멀리 하시고 의인의 기도를 들으시느니라잠15:30 눈이 밝은 것은 마음을 기쁘게 하고 좋은 기별은 뼈를 윤택하게 하느니라잠15:31 생명의 경계를 듣는 귀는 지혜로운 자 가운데에 있느니라


### Step 4 Retrievers

In [23]:
!pip install -U langchain langchain-classic
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
llm = ChatOpenAI(
    model="gpt-5",              # 또는 "gpt-5"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

In [24]:
query = "하나님은 누구신가"
result = qa(query)
from IPython.display import Markdown, display
display(Markdown(result["result"]))

/tmp/ipykernel_33038/122034310.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


질문에 주어진 잠언 말씀들이 보여 주는 하나님은 이런 분입니다.
- 악인을 멀리하시고 의인의 기도를 들으시는 분입니다 (잠언 15:29).
- 사람의 마음을 시험하고 정련하시는 분입니다 (잠언 17:3).
- 가난한 자의 창조주이시며, 약자를 조롱하는 것을 악으로 보시고 결코 무죄하지 않게 하시는 분입니다 (잠언 17:5).
- 반역과 악을 용납하지 않으시고 마땅한 심판이 따르게 하시는 분입니다 (잠언 17:11).
- 재물을 의지하는 길은 패망으로, 의로움의 길은 생명과 번성으로 이끄시는 질서를 세우신 분입니다 (잠언 11:28).

요약하면, 잠언이 증언하는 하나님은 의와 공의를 사랑하시며, 사람의 마음 깊은 곳까지 아시는 창조주이십니다.

질문에 주어진 잠언 말씀들이 보여 주는 하나님은 이런 분입니다.
- 악인을 멀리하시고 의인의 기도를 들으시는 분입니다 (잠언 15:29).
- 사람의 마음을 시험하고 정련하시는 분입니다 (잠언 17:3).
- 가난한 자의 창조주이시며, 약자를 조롱하는 것을 악으로 보시고 결코 무죄하지 않게 하시는 분입니다 (잠언 17:5).
- 반역과 악을 용납하지 않으시고 마땅한 심판이 따르게 하시는 분입니다 (잠언 17:11).
- 재물을 의지하는 길은 패망으로, 의로움의 길은 생명과 번성으로 이끄시는 질서를 세우신 분입니다 (잠언 11:28).

요약하면, 잠언이 증언하는 하나님은 의와 공의를 사랑하시며, 사람의 마음 깊은 곳까지 아시는 창조주이십니다.

### Step 5 Question Answering

In [27]:
query = "잘사는 사람이란?"
result = qa(query)


잠언에 따르면 “잘사는 사람”은 이런 사람입니다.
- 악을 멀리하고 의로워 하나님께서 기도를 들으시는 사람(잠15:29)
- 밝은 눈과 기쁜 마음으로, 좋은 소식으로 주변을 살리는 사람(잠15:30)
- 생명을 주는 책망을 기꺼이 듣고 지혜로운 자들과 함께하는 사람(잠15:31)
- 재물을 의지하지 않고 하나님을 의지하여 푸른 잎사귀처럼 번성하는 사람(잠11:28)
- 자기 집을 해치지 않고 지혜로 세우는 사람(잠11:29)
- 자신의 한계를 인정하고 거룩하신 이를 경외하는 사람(잠30:3-4)
- 하나님의 말씀을 순전히 믿고 더하지 않는 사람(잠30:5-6)
- 거짓과 헛됨을 멀리하고, 가난도 부요도 지나치지 않게 필요한 양식으로 만족을 구하는 사람(잠30:7-9)
- 배부름으로 교만하거나 궁핍으로 범죄하지 않기를 구해 하나님의 이름을 존중하는 사람(잠30:9)
- 아랫사람을 함부로 비방하지 않는 공의로운 사람(잠30:10)

요약하면, 잘사는 사람은 의와 지혜를 좇아 정직·절제·만족을 배우며, 하나님을 의지하고 이웃을 살리는 사람입니다.

잠언에 따르면 “잘사는 사람”은 이런 사람입니다.
- 악을 멀리하고 의로워 하나님께서 기도를 들으시는 사람(잠15:29)
- 밝은 눈과 기쁜 마음으로, 좋은 소식으로 주변을 살리는 사람(잠15:30)
- 생명을 주는 책망을 기꺼이 듣고 지혜로운 자들과 함께하는 사람(잠15:31)
- 재물을 의지하지 않고 하나님을 의지하여 푸른 잎사귀처럼 번성하는 사람(잠11:28)
- 자기 집을 해치지 않고 지혜로 세우는 사람(잠11:29)
- 자신의 한계를 인정하고 거룩하신 이를 경외하는 사람(잠30:3-4)
- 하나님의 말씀을 순전히 믿고 더하지 않는 사람(잠30:5-6)
- 거짓과 헛됨을 멀리하고, 가난도 부요도 지나치지 않게 필요한 양식으로 만족을 구하는 사람(잠30:7-9)
- 배부름으로 교만하거나 궁핍으로 범죄하지 않기를 구해 하나님의 이름을 존중하는 사람(잠30:9)
- 아랫사람을 함부로 비방하지 않는 공의로운 사람(잠30:10)

요약하면, 잘사는 사람은 의와 지혜를 좇아 정직·절제·만족을 배우며, 하나님을 의지하고 이웃을 살리는 사람입니다.

In [31]:
query = "좋은 친구란?"
result = qa(query)


성경 잠언이 말하는 “좋은 친구”는 이런 사람입니다.

- 함께 있을 때 마음을 기쁘게 하고 생기를 주는 사람 (잠15:30)
- 생명을 주는 책망과 지혜로운 권고를 해 주고, 그런 권고를 서로 듣는 사람 (잠15:31; 잠27:9)
- 환난 날에 곁을 지키며 관계를 버리지 않는 가까운 이웃 같은 사람 (잠27:10)
- 무모함을 말리고 재앙을 피하도록 경고해 주는 신중한 사람 (잠27:12)
- 요란하고 배려 없는 방식이 아니라, 때와 방법을 헤아리는 사람 (잠27:14)
- 서로를 연마하여 더 선하고 지혜롭게 자라게 하는 사람 (잠27:17)
- 의로움을 따르고 하나님께 가까이 나아가 기도하는 사람 (잠15:29)

한마디로, 좋은 친구는 곁을 지키며 지혜롭게 권면하고, 서로를 더 나은 방향으로 다듬어 주는 충성된 이웃입니다.

In [30]:
query = "좋은 배우자란?"
result = qa(query)

좋은 배우자는 함께 의와 진실을 사랑하고, 가정을 살리며, 하나님을 의지하는 사람입니다. 주신 말씀들로 보면 이런 모습입니다.

- 의를 추구하고 악에서 멀리함, 기도하는 사람 (잠 15:29)
- 따뜻한 눈빛과 말로 마음을 기쁘게 하고, 좋은 소식과 격려를 전함 (잠 15:30)
- 훈계와 교정을 기꺼이 듣는 배우려는 태도(가르침에 귀 기울임) (잠 15:31)
- 거짓과 헛됨을 멀리하는 정직함 (잠 30:8)
- 물질에 치우치지 않는 절제와 만족을 구함(부하기도 가난하기도보다 필요한 양식) (잠 30:8-9)
- 재물을 의지하지 않고 하나님을 의지함(하나님의 말씀을 신뢰하고 더하지 않음) (잠 30:5-6; 11:28)
- 집을 해치지 않고 세우는 책임감과 지혜 (잠 11:29)
- 남을 비방하지 않고 말로 죄를 만들지 않음 (잠 30:10)

분별을 위해 스스로와 서로에게 물어보면 좋습니다.
- 우리는 서로의 조언과 교정을 겸손히 듣는가? (잠 15:31)
- 돈보다 의와 하나님을 먼저 두는가? (잠 11:28; 30:8-9)
- 우리의 말이 상대를 살리고 격려하는가? (잠 15:30)
- 거짓과 비방을 멀리하는가? (잠 30:8,10)

완벽한 사람은 없지만, 이런 방향으로 함께 자라가려는 사람이 좋은 배우자입니다. 그리고 나 역시 그런 사람이 되려는 결심이 관계를 복되게 합니다.